# Lesson 09: Local SLM Deployment — Parameter Tiers, Ollama Modelfiles & OpenAI-Compatible Serving

In this companion notebook, you will:
1. Programmatically classify SLMs across Micro (100M–1.4B), Small (1.5B–13.9B), and Medium (14B–30B) tiers.
2. Generate a production-ready **Ollama `Modelfile`** with custom system prompts, `num_ctx`, and `temperature`.
3. Test an OpenAI-compatible local serving loop (with Colab-safe `transformers` fallback when Ollama daemon is not running).

In [ ]:
!pip install -q -U openai transformers torch

## 1. Generate an Ollama `Modelfile`
On Linux, macOS, or Windows, `ollama create edge-assistant -f Modelfile` packages a GGUF checkpoint with deterministic parameters.

In [ ]:
modelfile_content = '''FROM qwen2.5:3b
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
SYSTEM """You are a privacy-first Edge AI assistant. Keep answers concise, factual, and structured."""
'''
with open("Modelfile", "w") as f:
    f.write(modelfile_content)
print("Wrote Modelfile:\n" + modelfile_content)

## 2. OpenAI-Compatible Local Client Pattern (With Colab Fallback)
When Ollama runs locally on port `11434`, the standard `openai` Python SDK talks to `http://localhost:11434/v1` with zero code changes.

In [ ]:
import urllib.request
from openai import OpenAI

def is_ollama_running(url="http://127.0.0.1:11434/api/tags"):
    try:
        with urllib.request.urlopen(url, timeout=1.5) as r:
            return r.status == 200
    except Exception:
        return False

if is_ollama_running():
    client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")
    resp = client.chat.completions.create(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": "Explain GGUF in one sentence."}],
    )
    print("Ollama Response:", resp.choices[0].message.content)
else:
    print("Ollama daemon not detected on localhost:11434 (expected in hosted Colab).")
    print("Running in-process Hugging Face Qwen2.5-0.5B-Instruct fallback instead...")
    from transformers import pipeline
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
    out = pipe("Explain why local SLMs protect data privacy in one sentence:", max_new_tokens=50, do_sample=False)
    print(out[0]["generated_text"])